In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model

In [2]:
# Load the IMDB dataset word index
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

In [3]:
# Load the pre-trained model with ReLU activation
model = load_model('imdb_new.h5')
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,027 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [4]:
model.get_weights()

[array([[ 2.81986237e-01,  1.64410755e-01,  2.82076687e-01, ...,
         -3.33351679e-02,  2.35935360e-01,  4.30707604e-01],
        [ 5.14796702e-04,  7.60555342e-02, -6.15920164e-02, ...,
          4.60585169e-02,  4.44567613e-02, -3.56914178e-02],
        [ 1.75772235e-01,  6.17678799e-02,  4.32959683e-02, ...,
          1.59179285e-01,  9.56169665e-02, -1.11951925e-01],
        ...,
        [ 3.47459242e-02,  1.49294781e-02, -4.14658636e-02, ...,
          3.03852297e-02,  7.28427842e-02,  2.62541755e-04],
        [ 2.58853417e-02,  1.00437384e-02,  3.95905226e-02, ...,
         -1.08664948e-02,  6.30781101e-03,  2.41185017e-02],
        [-5.09690270e-02, -1.79559346e-02, -2.82898378e-02, ...,
          6.22933656e-02, -1.96857024e-02, -4.91251051e-03]],
       shape=(10000, 128), dtype=float32),
 array([[-0.03049493, -0.10305604, -0.0022143 , ...,  0.11074714,
         -0.0124116 ,  0.06825368],
        [ 0.10966342, -0.10358473, -0.02970295, ...,  0.0044003 ,
          0.1338749

In [5]:
# Function to decode reviews
def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

In [6]:
# Function to preprocess user input
def preprocess_text(text):
    max_features = 10000  # same value used during training
    
    words = text.lower().split()
    encoded_review = []

    for word in words:
        if word in word_index and word_index[word] < max_features:
            encoded_review.append(word_index[word] + 3)
        else:
            encoded_review.append(2)  
    padded_review = sequence.pad_sequences([encoded_review], maxlen=500)
    return padded_review

In [7]:
# Prediction  function

def predict_sentiment(review):
    preprocessed_input=preprocess_text(review)

    prediction=model.predict(preprocessed_input)

    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'
    
    return sentiment, prediction[0][0]


In [8]:
# Example review for prediction
example_review = "This movie was fantastic! The acting was great and the plot was thrilling."

sentiment,score=predict_sentiment(example_review)

print(f'Review: {example_review}')
print(f'Sentiment: {sentiment}')
print(f'Prediction Score: {score}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 282ms/step
Review: This movie was fantastic! The acting was great and the plot was thrilling.
Sentiment: Negative
Prediction Score: 0.40538591146469116
